# 08m THINGS-EEG1 test P-2: borrowing with a short recording (prespecified, scored once)

**Kaggle:** CPU, Internet on. **Inputs:** the two datasets of 08g, and the outputs of 08b and 08a (k-subsets only).

Protocol P-2 (`docs/proposals/eeg1_h8_protocol.md`), follow-up 2 of 3, alpha 0.05/3. Each reserved person is the new
person, borrowing from all 22 development people. Primary: the person's whitening estimated from only their k
labelled trials. Four contrasts (borrowing and Gu's ensemble minus the own ridge model and minus own ridge with an
output map), Bonferroni over four, margin 0.0035, at least 20 complete people.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '08m-h8confirm', 'cpu-confirm'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"sources": ["sub-04", "sub-07", "sub-11", "sub-12", "sub-13", "sub-14", "sub-15", "sub-17", "sub-20", "sub-21", "sub-24", "sub-25", "sub-26", "sub-29", "sub-30", "sub-34", "sub-35", "sub-38", "sub-39", "sub-46", "sub-48", "sub-49"], "targets": ["sub-02", "sub-03", "sub-05", "sub-08", "sub-09", "sub-10", "sub-16", "sub-19", "sub-22", "sub-27", "sub-28", "sub-31", "sub-32", "sub-33", "sub-36", "sub-37", "sub-40", "sub-41", "sub-42", "sub-43", "sub-44", "sub-45", "sub-47", "sub-50"], "min_targets": 20, "budgets": [250, 500], "draws": [13, 14, 15], "conditions": ["k", 2000, "all"], "primary": "k", "delta": 0.003547826037718968, "alpha": 0.016666666666666666, "window_s": [0.1, 0.2], "feature_sha256": "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f", "k_subsets_sha256": "3f3ede66115c9fefd34e15d0b93a5fc10ef3f12ed32e251a3aab1a79732e01c1"}')
import numpy as np
import pandas as pd
from scipy import stats

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.data import things_eeg2 as e2
from cross_subject_encoding.eval import eeg1_h8 as h8
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import window_groups
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
from cross_subject_encoding.utils.kaggle_export import export_to_dataset
from cross_subject_encoding.utils.preflight import preflight

preflight("cross-subject-brain-encoding-08m-outputs", inputs=["eeg1_paper_kpca1000.npz", "eeg1_k_subsets.json"],
          need_gpu=False, expect_output_gb=0.2)
feature_path, subsets_path = C.find_artifact("eeg1_paper_kpca1000.npz"), C.find_artifact("eeg1_k_subsets.json")
assert sha256_file(feature_path) == FIXED["feature_sha256"] and sha256_file(subsets_path) == FIXED["k_subsets_sha256"]
subsets = json.loads(subsets_path.read_text())
with np.load(feature_path) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
sources, targets = FIXED["sources"], FIXED["targets"]
def root_of(tag):
    hits = [p for p in Path("/kaggle/input").rglob("sub-*_preprocessed_eeg_training.npy") if tag in str(p)]
    assert hits, f"attach the {tag} dataset"
    return hits[0].parent
ROOTS = {"08g-sources": root_of("08g-sources"), "08g-targets": root_of("08g-targets")}
opened = []

def load(person, part):
    root = ROOTS["08g-sources"] if person in sources else ROOTS["08g-targets"]
    path = source_file(root, person, part, sources + targets)
    opened.append(path.name)
    release = e2.load_part(path)
    assert list(release["ch_names"]) == e1.CHANNELS
    if not np.isfinite(release["data"]).all():
        raise ValueError(f"{person}/{part}: non-finite values; needs the millivolt version of 08g")
    return release["data"], release["times"]

raw = {}
for s in sources + targets:
    (tr, times), (te, _) = load(s, "training"), load(s, "test")
    raw[s] = (tr[:, 0], te)
src_train = np.empty((len(sources), e1.N_TRAIN, 17 * 100), np.float32)
src_test = np.empty((len(sources), e1.N_TEST, 17 * 100), np.float32)
query = np.concatenate([z_train, z_test])
for i, s in enumerate(sources):  # sources stay fully whitened (all their trials)
    tr, te = h8.whiten(raw[s][0], raw[s][1], np.arange(e1.N_TRAIN))
    pred = pl.ols_paper(z_train, tr.reshape(e1.N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:e1.N_TRAIN], pred[e1.N_TRAIN:]
groups = window_groups(times, 17, C.EEG_WINDOW_MS)
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
print("loaded and fitted sources |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
rows, failed = [], {}
for person in targets:
    t = time.monotonic()
    try:
        orders = {d: subsets["orders"][person][str(d)] for d in FIXED["draws"]}
        out = h8.evaluate_person(raw[person][0], raw[person][1], src_train, src_test, z_train, z_test, orders,
                                 FIXED["budgets"], FIXED["conditions"], groups, ALPHAS, C.RIDGE_FOLDS)
        for (condition, method, k, draw), curve in out.items():
            if not np.isfinite(curve).all():
                raise ValueError(f"nonfinite score {method}")
            rows.append({"person": person, "condition": condition, "method": method, "k": k, "draw": draw,
                         "r_100_200ms": float(curve[:, window].mean())})
    except Exception as error:  # P-2: a failed target is excluded and reported, never imputed
        failed[person] = f"{type(error).__name__}: {error}"
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "08m_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s", failed.get(person, ""))
frame = pd.DataFrame(rows)
done = sorted(set(frame["person"])) if len(frame) else []
mean = frame.groupby(["person", "condition", "method", "k"])["r_100_200ms"].mean() if len(frame) else {}
ws = {key: value for key, value in mean.items()}
summary = {"people": len(done), "failed": failed, "eeg_files_opened": len(opened), "conditions": {}}
t_crit = stats.t.ppf(1 - FIXED["alpha"] / len(h8.CONTRASTS), max(len(done) - 1, 1))
for condition in map(str, FIXED["conditions"]):
    d = h8.contrasts(ws, done, condition, FIXED["budgets"])
    res = {}
    for j, (b_, o) in enumerate(h8.CONTRASTS):
        col = d[:, j]
        half = t_crit * col.std(ddof=1) / np.sqrt(len(col))
        res[f"{b_} - {o}"] = {"mean": float(col.mean()), "lower": float(col.mean() - half), "upper": float(col.mean() + half),
                              "positive": int((col > 0).sum())}
    if len(done) < FIXED["min_targets"]:
        verdict = "insufficient"
    else:
        lows, ups = [v["lower"] for v in res.values()], [v["upper"] for v in res.values()]
        verdict = "supported" if min(lows) > FIXED["delta"] else "rejected" if min(ups) < FIXED["delta"] else "inconclusive"
    summary["conditions"][condition] = {"contrasts": res, "verdict": verdict,
                                        "method_means": {f"{m}|{k}": float(np.mean([ws[(p, condition, m, k)] for p in done]))
                                                         for m in h8.BORROWING + h8.OWN for k in FIXED["budgets"]}}
summary["primary_verdict"] = summary["conditions"][str(FIXED["primary"])]["verdict"]
summary["t_critical"] = float(t_crit)
(C.METRICS_DIR / "08m_decision.json").write_text(json.dumps(summary, indent=2))
print("PRIMARY VERDICT (m = k):", summary["primary_verdict"])
print(json.dumps(summary, indent=1)[:3000])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-08m-outputs", title="XSUB 08m H8 confirmation outputs",
                  files=[C.METRICS_DIR / "08m_decision.json", C.METRICS_DIR / "08m_scores.csv"], include_dirs=[],
                  message="h8 confirmation")